# Results Analysis
Loads `checkpoint_evals.json` for all conditions and plots training trajectories.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path().resolve().parent
RESULTS = ROOT / 'results'

CONDITIONS = ['flat', 'chunked', 'balanced']
LABELS = {'flat': 'Sliding-Window', 'chunked': 'EOS-Chunked', 'balanced': 'DENSITY-Balanced'}
COLORS = {'flat': '#E64B35', 'chunked': '#4DBBD5', 'balanced': '#00A087'}

# Published reference: BabyLM-community/babylm-baseline-10m-gpt2 (HuggingFace)
BABYLM_BLIMP_REFERENCE = 0.6636

def _results_dir(cond):
    """Prefer {cond}_baseline/ (new 10M runs), fall back to {cond}/ (old pilot)."""
    for name in [f'{cond}_baseline', cond]:
        d = RESULTS / name
        if d.exists():
            return d
    return None

def load_evals(cond):
    d = _results_dir(cond)
    if d is None: return []
    p = d / 'checkpoint_evals.json'
    if not p.exists(): return []
    data = json.loads(p.read_text())
    ckpts = [c for c in data.get('checkpoints', []) if c.get('words_seen')]
    return sorted(ckpts, key=lambda c: c['words_seen'])

def load_curves(cond):
    """Load training_curves.json — returns dict or None."""
    d = _results_dir(cond)
    if d is None: return None
    p = d / 'training_curves.json'
    if not p.exists(): return None
    return json.loads(p.read_text())

evals = {c: load_evals(c) for c in CONDITIONS}
curves = {c: load_curves(c) for c in CONDITIONS}
for c in CONDITIONS:
    src = _results_dir(c)
    print(f'{c}: {len(evals[c])} eval checkpoints | curves: {"yes" if curves[c] else "no"} | dir: {src.name if src else "not found"}')

## Final checkpoint numbers

In [ ]:
rows = []
for cond in CONDITIONS:
    ckpts = evals[cond]
    if not ckpts: continue
    last = ckpts[-1]
    blimp_ckpts = [c for c in ckpts if 'blimp' in c]
    rows.append({
        'Condition':    LABELS[cond],
        'Words Seen':   f"{last['words_seen']/1e6:.1f}M",
        'FBT Overall':  round(last.get('fbt_overall',      float('nan')), 3),
        'False Belief': round(last.get('fbt_false_belief', float('nan')), 3),
        'True Belief':  round(last.get('fbt_true_belief',  float('nan')), 3),
        'BLiMP':        round(blimp_ckpts[-1]['blimp'], 3) if blimp_ckpts else float('nan'),
        'EWoK':         round([c for c in ckpts if 'ewok' in c][-1]['ewok'], 3)
                        if any('ewok' in c for c in ckpts) else float('nan'),
    })

if rows:
    pd.DataFrame(rows).set_index('Condition')
else:
    print('No checkpoint data found — run eval_checkpoints.py first.')

## FBT trajectories — False Belief Trap

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
fig.suptitle('FBT Accuracy During Training', fontsize=13, fontweight='bold')

for ax, cond in zip(axes, CONDITIONS):
    ckpts = evals[cond]
    if not ckpts:
        ax.set_title(f'{LABELS[cond]}\n(no data)')
        continue
    xs  = np.array([c['words_seen'] / 1e6 for c in ckpts])
    fbt = np.array([c['fbt_overall']       for c in ckpts], dtype=float)
    fb  = np.array([c['fbt_false_belief']  for c in ckpts], dtype=float)
    tb  = np.array([c['fbt_true_belief']   for c in ckpts], dtype=float)

    ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
    ax.plot(xs, fbt, color='#3C5488', lw=2,   label='Overall')
    ax.plot(xs, fb,  color='#E64B35', lw=1.5, label='False Belief')
    ax.plot(xs, tb,  color='#4DBBD5', lw=1.5, label='True Belief')

    ax.set_title(LABELS[cond], fontsize=11, fontweight='bold')
    ax.set_xlabel('Words Seen (M)')
    ax.set_ylim(0.10, 1.0)
    for sp in ['top','right']: ax.spines[sp].set_visible(False)

axes[0].set_ylabel('P(Correct)')
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

## BLiMP — syntactic competence

In [ ]:
all_blimp = [c['blimp'] for cond in CONDITIONS for c in evals[cond] if 'blimp' in c]
ylim_blimp = (max(0.0, min(all_blimp + [BABYLM_BLIMP_REFERENCE]) - 0.03),
              min(1.0, max(all_blimp + [BABYLM_BLIMP_REFERENCE]) + 0.03)) if all_blimp else (0.45, 0.75)

fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
ax.axhline(BABYLM_BLIMP_REFERENCE, color='#333', lw=1, ls=':',
           label=f'BabyLM Baseline ({BABYLM_BLIMP_REFERENCE:.2%})')

for cond in CONDITIONS:
    ckpts = [c for c in evals[cond] if 'blimp' in c]
    if not ckpts: continue
    xs = np.array([c['words_seen'] / 1e6 for c in ckpts])
    ys = np.array([c['blimp']            for c in ckpts], dtype=float)
    ax.plot(xs, ys, color=COLORS[cond], lw=2, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('BLiMP Accuracy')
ax.set_title('BLiMP Over Training', fontsize=12, fontweight='bold')
ax.set_ylim(*ylim_blimp)
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## EWoK — world knowledge (null result)

In [ ]:
all_ewok = [c['ewok'] for cond in CONDITIONS for c in evals[cond] if 'ewok' in c]
ylim_ewok = (max(0.0, min(all_ewok) - 0.02), min(1.0, max(all_ewok) + 0.02)) if all_ewok else (0.44, 0.56)

fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')

DASH = {'flat': '-', 'chunked': '--', 'balanced': ':'}
for cond in CONDITIONS:
    ckpts = [c for c in evals[cond] if 'ewok' in c]
    if not ckpts: continue
    xs = np.array([c['words_seen'] / 1e6 for c in ckpts])
    ys = np.array([c['ewok']             for c in ckpts], dtype=float)
    ax.plot(xs, ys, color=COLORS[cond], lw=2, ls=DASH[cond], label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('EWoK Accuracy')
ax.set_title('EWoK Over Training — all conditions at chance', fontsize=12, fontweight='bold')
ax.set_ylim(*ylim_ewok)
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## False Belief Bias (FB − TB gap)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.axhline(0, color='#999', lw=1, ls='--', label='No bias')

for cond in CONDITIONS:
    ckpts = evals[cond]
    if not ckpts:
        continue
    xs  = np.array([c['words_seen'] / 1e6 for c in ckpts])
    gap = np.array([c['fbt_false_belief'] - c['fbt_true_belief'] for c in ckpts], dtype=float)
    ax.plot(xs, gap, color=COLORS[cond], lw=2, label=LABELS[cond])

ax.set_xlabel('Words Seen (M)')
ax.set_ylabel('FB − TB')
ax.set_title('False Belief Bias Over Training', fontsize=12, fontweight='bold')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Per-item breakdown — final checkpoint

In [ ]:
# Condition × Cue type breakdown from checkpoint_evals
rows = []
for cond in CONDITIONS:
    ckpts = evals[cond]
    if not ckpts: continue
    last = ckpts[-1]
    for key, val in last.items():
        if key.startswith('fbt_') and isinstance(val, float):
            rows.append({'condition': LABELS[cond], 'metric': key.replace('fbt_',''), 'value': round(val, 3)})

df = pd.DataFrame(rows)
df.pivot(index='metric', columns='condition', values='value')

## Training curves — per condition
One figure per condition: FBT (Overall / False Belief / True Belief), BLiMP, and EWoK trajectories side by side.

In [ ]:
for cond in CONDITIONS:
    cv = curves[cond]
    ck = evals[cond]
    if cv is None and not ck:
        print(f'{LABELS[cond]}: no data, skipping')
        continue

    color = COLORS[cond]
    fig, (ax_loss, ax_fbt, ax_blimp) = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(LABELS[cond], fontsize=13, fontweight='bold')

    # --- Loss panel (from training_curves.json) ---
    if cv:
        ws  = np.array(cv['words_seen']) / 1e6
        tl  = np.array(cv['train_loss'],  dtype=float)
        el  = np.array(cv['eval_loss'],   dtype=float)
        ax_loss.plot(ws, tl, color=color,   lw=1.5, alpha=0.7, label='Train loss')
        ax_loss.plot(ws, el, color='#333',  lw=2,               label='Eval loss')
        if cv.get('early_stopped_at'):
            stop_w = ws[cv['step'].index(cv['early_stopped_at'])] if cv['early_stopped_at'] in cv['step'] else None
            if stop_w:
                ax_loss.axvline(stop_w, color='#999', lw=1, ls=':', label='Early stop')
        all_loss = np.concatenate([tl, el])
        ax_loss.set_ylim(max(0, all_loss.min() - 0.2), all_loss.max() + 0.2)
        ax_loss.legend(fontsize=8)
    else:
        ax_loss.text(0.5, 0.5, 'No loss data', ha='center', va='center',
                     transform=ax_loss.transAxes, color='#aaa')
    ax_loss.set_title('Loss')
    ax_loss.set_xlabel('Words Seen (M)')
    ax_loss.set_ylabel('Cross-entropy Loss')
    for sp in ['top', 'right']: ax_loss.spines[sp].set_visible(False)

    # --- FBT panel (from checkpoint_evals.json) ---
    if ck:
        xs  = np.array([c['words_seen'] / 1e6 for c in ck])
        fbt = np.array([c['fbt_overall']       for c in ck], dtype=float)
        fb  = np.array([c['fbt_false_belief']  for c in ck], dtype=float)
        tb  = np.array([c['fbt_true_belief']   for c in ck], dtype=float)
        ax_fbt.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
        ax_fbt.plot(xs, fbt, color='#3C5488', lw=2,   label='Overall')
        ax_fbt.plot(xs, fb,  color='#E64B35', lw=1.5, label='False Belief')
        ax_fbt.plot(xs, tb,  color='#4DBBD5', lw=1.5, label='True Belief')
        fbt_vals = np.concatenate([fbt, fb, tb])
        ax_fbt.set_ylim(max(0.0, fbt_vals.min() - 0.05), min(1.0, fbt_vals.max() + 0.05))
        ax_fbt.legend(fontsize=8)
    else:
        ax_fbt.text(0.5, 0.5, 'No FBT data', ha='center', va='center',
                    transform=ax_fbt.transAxes, color='#aaa')
    ax_fbt.set_title('FBT Accuracy')
    ax_fbt.set_xlabel('Words Seen (M)')
    ax_fbt.set_ylabel('P(Correct)')
    for sp in ['top', 'right']: ax_fbt.spines[sp].set_visible(False)

    # --- BLiMP panel (from checkpoint_evals.json) ---
    blimp_ckpts = [c for c in ck if 'blimp' in c]
    if blimp_ckpts:
        bx = np.array([c['words_seen'] / 1e6 for c in blimp_ckpts])
        by = np.array([c['blimp']             for c in blimp_ckpts], dtype=float)
        ax_blimp.axhline(0.5, color='#999', lw=1, ls='--', label='Chance')
        ax_blimp.axhline(BABYLM_BLIMP_REFERENCE, color='#333', lw=1, ls=':',
                         label=f'BabyLM ref ({BABYLM_BLIMP_REFERENCE:.2%})')
        ax_blimp.plot(bx, by, color=color, lw=2)
        all_by = list(by) + [BABYLM_BLIMP_REFERENCE]
        ax_blimp.set_ylim(max(0.0, min(all_by) - 0.03), min(1.0, max(all_by) + 0.03))
        ax_blimp.legend(fontsize=8)
    else:
        ax_blimp.text(0.5, 0.5, 'No BLiMP data', ha='center', va='center',
                      transform=ax_blimp.transAxes, color='#aaa')
    ax_blimp.set_title('BLiMP')
    ax_blimp.set_xlabel('Words Seen (M)')
    ax_blimp.set_ylabel('Accuracy')
    for sp in ['top', 'right']: ax_blimp.spines[sp].set_visible(False)

    plt.tight_layout()
    plt.show()